In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

import sys
sys.path.append("..")

from src.backtester import calculate_return, backtester

In [2]:
df = pd.read_csv("../data/spy_daily.csv")
prices = df["Adj Close"]
returns = calculate_return(prices)

In [6]:
def backtester_v2(returns, signal, transaction_cost):

    df = pd.DataFrame({
        "return": returns,
        "signal": signal
    })

    df["position"] = df["signal"].shift(1)
    df["strategy_return"] = df["return"] * df["position"]


    # Time Invested

    days_invested = (df["position"] == 1).sum()
    valid_days = df["position"].count()
    time_invested = days_invested / valid_days


    # Trade Counts

    position = df["position"].fillna(0).tolist()
    trade_indicator = [0]
    
    for i in range(1, len(position)):
        if position[i] != position[i-1]:
            trade_indicator.append(1)
        else:
            trade_indicator.append(0)
    
    df["trade"] = trade_indicator
    trade_count = df["trade"].sum()


    # Transaction Costs

    df["transaction_cost"] = df["trade"] * transaction_cost
    df["net_strategy_return"] = df["strategy_return"] - df["transaction_cost"]


    # Equity

    performance_return = df["net_strategy_return"]

    first_valid = performance_return.first_valid_index()

    if first_valid is None:
        raise ValueError("No valid strategy returns were generated.")

    initial_equity = 1.0

    df["equity"] = np.nan
    start_index = first_valid - 1

    df.loc[start_index, "equity"] = initial_equity
    
    valid_return = performance_return.dropna()
    compounded_equity = (1 + valid_return).cumprod()
    
    df.loc[compounded_equity.index, "equity"] = compounded_equity

    final_equity = float(df["equity"].dropna().iloc[-1])

    total_return = (final_equity / initial_equity) - 1


    # Volatility + Sharpe Ratio

    risk_free_rate = 0
    
    daily_volatility = performance_return.std()
    mean_daily_return = performance_return.mean()
    sharpe_daily = (mean_daily_return - risk_free_rate)/daily_volatility
    
    annual_volatility = np.sqrt(252) * daily_volatility
    sharpe_ratio = np.sqrt(252) * sharpe_daily


    # Drawdown

    df["running_peak"] = np.nan

    df.loc[start_index, "running_peak"] = initial_equity
    
    for i in range(1, len(df["equity"])):
        df.loc[i, "running_peak"] = max(df.loc[i, "equity"], df.loc[i-1, "running_peak"])
    
    df["drawdown"] = df["equity"] / df["running_peak"] - 1
    max_drawdown = df["drawdown"].min()


    # CAGR + Calmar Ratio

    growth_multiple = final_equity / initial_equity
    years = performance_return.count() / 252
    cagr = growth_multiple ** (1 / years) - 1
    calmar_ratio = cagr / abs(max_drawdown)


    results = {
        "total_return": total_return,
        "cagr": cagr,
        "annual_volatility": annual_volatility,
        "annual_sharpe": sharpe_ratio,
        "max_drawdown": max_drawdown,
        "calmar": calmar_ratio,
        "time_invested": time_invested,
        "trades": trade_count
    }

    return df, results
    

In [7]:
def backtester_v3(returns, signal, transaction_cost):

    df = pd.DataFrame({
        "return": returns,
        "signal": signal
    })

    df["position"] = df["signal"].shift(1)
    df["strategy_return"] = df["return"] * df["position"]


    # Time Invested

    days_invested = (df["position"] == 1).sum()
    valid_days = df["position"].count()
    time_invested = days_invested / valid_days


    # Trade Counts

    position = df["position"].fillna(0).tolist()
    trade_indicator = [0]
    
    for i in range(1, len(position)):
        if position[i] != position[i-1]:
            trade_indicator.append(1)
        else:
            trade_indicator.append(0)
    
    df["trade"] = trade_indicator
    trade_count = df["trade"].sum()


    # Transaction Costs

    df["transaction_cost"] = df["trade"] * transaction_cost
    df["net_strategy_return"] = df["strategy_return"] - df["transaction_cost"]


    # Equity

    performance_return = df["net_strategy_return"]

    first_valid = performance_return.first_valid_index()

    if first_valid is None:
        raise ValueError("No valid strategy returns were generated.")

    initial_equity = 1.0

    df["equity"] = np.nan
    start_index = first_valid - 1

    df.loc[start_index, "equity"] = initial_equity
    
    valid_return = performance_return.dropna()
    compounded_equity = (1 + valid_return).cumprod()
    
    df.loc[compounded_equity.index, "equity"] = compounded_equity

    final_equity = float(df["equity"].dropna().iloc[-1])

    total_return = (final_equity / initial_equity) - 1


    # Volatility + Sharpe Ratio

    risk_free_rate = 0
    
    daily_volatility = performance_return.std()
    mean_daily_return = performance_return.mean()
    sharpe_daily = (mean_daily_return - risk_free_rate)/daily_volatility
    
    annual_volatility = np.sqrt(252) * daily_volatility
    sharpe_ratio = np.sqrt(252) * sharpe_daily


    # Drawdown

    df["running_peak"] = df["equity"].cummax()
    
    df["drawdown"] = df["equity"] / df["running_peak"] - 1
    max_drawdown = df["drawdown"].min()


    # CAGR + Calmar Ratio

    growth_multiple = final_equity / initial_equity
    years = performance_return.count() / 252
    cagr = growth_multiple ** (1 / years) - 1
    calmar_ratio = cagr / abs(max_drawdown)


    results = {
        "total_return": total_return,
        "cagr": cagr,
        "annual_volatility": annual_volatility,
        "annual_sharpe": sharpe_ratio,
        "max_drawdown": max_drawdown,
        "calmar": calmar_ratio,
        "time_invested": time_invested,
        "trades": trade_count
    }

    return df, results
    

In [3]:
def backtester_v4(returns, signal, transaction_cost):

    df = pd.DataFrame({
        "return": returns,
        "signal": signal
    })

    df["position"] = df["signal"].shift(1)
    df["strategy_return"] = df["return"] * df["position"]


    # Time Invested

    days_invested = (df["position"] == 1).sum()
    valid_days = df["position"].count()
    time_invested = days_invested / valid_days


    # Trade Counts

    position = df["position"].fillna(0)
    previous_position = position.shift(1).fillna(0)

    trade_indicator = (position != previous_position).astype(int)
    trade_count = trade_indicator.sum()


    # Transaction Costs

    df["transaction_cost"] = trade_indicator * transaction_cost
    df["net_strategy_return"] = df["strategy_return"] - df["transaction_cost"]


    # Equity

    performance_return = df["net_strategy_return"]

    first_valid = performance_return.first_valid_index()

    if first_valid is None:
        raise ValueError("No valid strategy returns were generated.")

    initial_equity = 1.0

    df["equity"] = np.nan
    start_index = first_valid - 1

    df.loc[start_index, "equity"] = initial_equity
    
    valid_return = performance_return.dropna()
    compounded_equity = (1 + valid_return).cumprod()
    
    df.loc[compounded_equity.index, "equity"] = compounded_equity

    final_equity = float(df["equity"].dropna().iloc[-1])

    total_return = (final_equity / initial_equity) - 1


    # Volatility + Sharpe Ratio

    risk_free_rate = 0
    
    daily_volatility = performance_return.std()
    mean_daily_return = performance_return.mean()
    sharpe_daily = (mean_daily_return - risk_free_rate)/daily_volatility
    
    annual_volatility = np.sqrt(252) * daily_volatility
    sharpe_ratio = np.sqrt(252) * sharpe_daily


    # Drawdown

    df["running_peak"] = df["equity"].cummax()
    
    df["drawdown"] = df["equity"] / df["running_peak"] - 1
    max_drawdown = df["drawdown"].min()


    # CAGR + Calmar Ratio

    growth_multiple = final_equity / initial_equity
    years = performance_return.count() / 252
    cagr = growth_multiple ** (1 / years) - 1
    calmar_ratio = cagr / abs(max_drawdown)


    results = {
        "total_return": total_return,
        "cagr": cagr,
        "annual_volatility": annual_volatility,
        "annual_sharpe": sharpe_ratio,
        "max_drawdown": max_drawdown,
        "calmar": calmar_ratio,
        "time_invested": time_invested,
        "trades": trade_count
    }

    return df, results
    

#### Elementary Random Benchmark

In [8]:
def generate_random_signal(prices, lookback):

    random_signal = [np.nan] * lookback

    for i in range(lookback, len(prices)):
        choice = np.random.choice([0,1])

        random_signal.append(choice)

    return random_signal

In [4]:
def generate_random_signal_v2(prices, lookback):

    random_signal = [np.nan] * lookback

    choice = np.random.choice([0,1], size = len(prices) - lookback)

    random_signal = random_signal + choice.tolist()

    return random_signal

In [5]:
df["Random Signal"] = generate_random_signal_v2(prices, 252)
random_signal = df["Random Signal"]

In [7]:
random_df, random_result = backtester_v4(returns, random_signal, 0.001)
pd.Series(random_result)

total_return           -0.865213
cagr                   -0.075199
annual_volatility       0.136703
annual_sharpe          -0.503658
max_drawdown           -0.885275
calmar                 -0.084944
time_invested           0.493808
trades               3225.000000
dtype: float64

In [8]:
def run_simulation(n):

    simulation_result = []

    for i in range(n):

        random_signal = generate_random_signal_v2(prices, 252)
        _, random_result = backtester_v4(returns, random_signal, 0.001)

        simulation_result.append(random_result)

    return simulation_result



In [11]:
simulation_output = run_simulation(20000)
simulation_df = pd.DataFrame(simulation_output)
simulation_df

,total_return,cagr,annual_volatility,annual_sharpe,max_drawdown,calmar,time_invested,trades
0,-0.877798,-0.078729,0.135476,-0.537087,-0.895731,-0.087893,0.503406,3201
1,-0.819071,-0.064517,0.130034,-0.447580,-0.864203,-0.074655,0.493963,3160
2,-0.939471,-0.103635,0.139060,-0.716689,-0.948541,-0.109257,0.502786,3223
3,-0.855794,-0.072759,0.134437,-0.494381,-0.907059,-0.080214,0.503251,3248
4,-0.980603,-0.142556,0.129424,-1.122650,-0.982603,-0.145080,0.491486,3271
...,...,...,...,...,...,...,...,...
19995,-0.931937,-0.099523,0.138078,-0.690006,-0.940615,-0.105806,0.497988,3214
19996,-0.876557,-0.078365,0.139186,-0.516674,-0.886121,-0.088436,0.506656,3158
19997,-0.790174,-0.059094,0.137062,-0.375975,-0.862462,-0.068518,0.499071,3211
19998,-0.930702,-0.098891,0.132938,-0.716792,-0.932825,-0.106012,0.493344,3226
